# 47 — DeepEval RAG Metrics

**What you'll learn:**
- All 5 RAG-specific DeepEval metrics with their mathematical definitions
- How to construct `LLMTestCase` objects from retrieval results
- Running `deepeval.evaluate()` on a golden dataset
- Injecting deliberate failures (hallucinated answers) and watching scores drop
- Interpreting score breakdowns and per-metric reasons

**Context:** DeepEval (Confident AI, 2024) provides LLM-powered metrics that correlate significantly better with human judgment than lexical metrics like ROUGE or BLEU. Each metric uses a small LLM judge to decompose and evaluate answers rather than relying on string overlap.

In [ ]:
%pip install -q deepeval langchain langchain-openai langchain-chroma chromadb python-dotenv

In [ ]:
import os

# Colab: uncomment and set your key
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Local: load from .env file
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"
print("API key loaded.")

In [ ]:
# ── DeepEval RAG Metric Definitions ──────────────────────────────────────────
#
# 1. Faithfulness = supported_claims / total_claims
#    - LLM breaks answer into claims, checks each against retrieved context
#    - Score 1.0 = every claim is grounded; 0.0 = all hallucinated
#
# 2. Answer Relevancy = relevant_sentences / total_sentences
#    - Measures how much of the answer actually addresses the question
#    - Score 1.0 = fully on-topic; 0.0 = complete non-answer
#
# 3. Contextual Precision = Σ(precision@k * relevance_k) / relevant_count
#    - Are the MOST relevant chunks ranked FIRST in retrieval?
#    - Score 1.0 = perfect ranking; penalizes relevant chunks buried at k=5
#
# 4. Contextual Recall = expected_output_covered / total_expected_content
#    - How much of the expected answer is supported by retrieved context?
#    - Score 1.0 = full coverage; measures retrieval completeness
#
# 5. Contextual Relevancy = relevant_chunks / total_chunks
#    - What fraction of retrieved chunks are actually relevant?
#    - Score 1.0 = all retrieved chunks useful; 0.0 = pure noise

print("Metric summary loaded.")
print("Faithfulness + AnswerRelevancy  → generation quality")
print("ContextualPrecision + ContextualRecall + ContextualRelevancy  → retrieval quality")

In [ ]:
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

KNOWLEDGE_BASE = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based workflows.",
    "LangChain provides tools for building LLM applications, including chains, agents, and retrieval systems.",
    "RAG (Retrieval-Augmented Generation) combines vector search with LLM generation to reduce hallucinations.",
    "ChromaDB is an open-source vector database for storing and retrieving embeddings at local scale.",
    "Embeddings are dense numerical representations of text that capture semantic meaning.",
    "FAISS is Facebook's library for efficient similarity search over large vector datasets.",
    "Faithfulness measures whether an LLM answer is grounded in the retrieved context (no hallucinations).",
    "Answer Relevancy measures whether the answer actually addresses the user's question.",
    "Contextual Precision measures whether the retrieved chunks that appear earlier are more relevant than later ones.",
    "Contextual Recall measures how much of the expected answer is covered by the retrieved context.",
]

GOLDEN_DATASET = [
    {
        "input": "What is LangGraph?",
        "expected_output": "LangGraph is a library for building stateful, multi-actor LLM applications using graph-based workflows.",
        "context": [
            "LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based workflows."
        ],
    },
    {
        "input": "What does Faithfulness measure?",
        "expected_output": "Faithfulness measures whether the LLM answer is grounded in the retrieved context, preventing hallucinations.",
        "context": [
            "Faithfulness measures whether an LLM answer is grounded in the retrieved context (no hallucinations)."
        ],
    },
    {
        "input": "What is ChromaDB used for?",
        "expected_output": "ChromaDB is an open-source vector database for storing and querying embeddings locally.",
        "context": [
            "ChromaDB is an open-source vector database for storing and retrieving embeddings at local scale."
        ],
    },
]

ANSWER_PROMPT = """Answer the question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know.'

Context:
{context}

Question: {query}

Answer:"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(KNOWLEDGE_BASE, embeddings, collection_name="deepeval_nb")

rag_results = []
for case in GOLDEN_DATASET:
    docs = vectorstore.similarity_search(case["input"], k=3)
    context = [d.page_content for d in docs]
    context_text = "\n".join(f"- {c}" for c in context)
    prompt = ANSWER_PROMPT.format(context=context_text, query=case["input"])
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    rag_results.append({"query": case["input"], "answer": answer, "context": context})
    print(f"Q: {case['input']}")
    print(f"A: {answer[:150]}\n")

print(f"Collected {len(rag_results)} results for evaluation.")

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)
from deepeval.test_case import LLMTestCase

metrics = [
    FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
    ContextualPrecisionMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
    ContextualRecallMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
    ContextualRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
]

# Build test cases from golden dataset results
test_cases = [
    LLMTestCase(
        input=case["input"],
        actual_output=result["answer"],
        expected_output=case["expected_output"],
        retrieval_context=result["context"],
    )
    for case, result in zip(GOLDEN_DATASET, rag_results)
]

print("Running evaluation on grounded answers...")
results = evaluate(test_cases, metrics)
print("Evaluation complete. Check scores above.")

In [ ]:
# ── Failure Injection ─────────────────────────────────────────────────────────
# Replace correct answers with hallucinated ones and re-run the same metrics.
# Watch Faithfulness and AnswerRelevancy scores drop significantly.

HALLUCINATED_ANSWERS = [
    "LangGraph is a cloud service by OpenAI that costs $99/month.",
    "Faithfulness measures the length of the answer in words.",
    "ChromaDB requires a PostgreSQL database to function.",
]

hallucinated_test_cases = [
    LLMTestCase(
        input=case["input"],
        actual_output=hallucinated,
        expected_output=case["expected_output"],
        retrieval_context=result["context"],
    )
    for case, result, hallucinated in zip(GOLDEN_DATASET, rag_results, HALLUCINATED_ANSWERS)
]

print("Running evaluation on hallucinated answers...")
hallucinated_results = evaluate(hallucinated_test_cases, metrics)
print("Evaluation complete.\n")

# Score comparison reminder
print(f"{'Metric':<28} {'Grounded':>10} {'Hallucinated':>14}")
print("-" * 56)
metric_names = [
    "Faithfulness",
    "AnswerRelevancy",
    "ContextualPrecision",
    "ContextualRecall",
    "ContextualRelevancy",
]
for name in metric_names:
    print(f"{name:<28} {'(above)':>10} {'(above)':>14}")
print(
    "\nCompare the score outputs from cells 6 and 7 — hallucinated answers score lower on Faithfulness and AnswerRelevancy."
)

## Exercises

1. **Retrieval depth:** Change `k=3` to `k=1` in cell 5. Measure the drop in Contextual Recall — fewer retrieved chunks means less coverage of the expected answer.

2. **Context poisoning:** Inject a wrong fact into `KNOWLEDGE_BASE` (e.g., change the LangGraph description). Re-run and watch Faithfulness fall below the 0.7 threshold.

3. **Custom golden dataset:** Write your own 5-question golden dataset from entries in `KNOWLEDGE_BASE`. Run all 5 metrics. Which metric is hardest to score above 0.9?

4. **Threshold sensitivity:** Set `threshold=0.9` across all metrics in cell 6. Which metric fails first on grounded answers? This reveals the weakest link in this pipeline.

---

**What's next:**
- `48-deepeval-hallucination-bias` — safety-oriented metrics (Hallucination, Bias, Toxicity)
- `29-llm-judge-harness` — build a custom LLM judge without DeepEval
- `16-rag-eval-notebook` — RAGAS comparison: how DeepEval and RAGAS scores differ on the same pipeline